#1. Install PySpark


In [1]:
pip install pyspark

2. Start Spark Session

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark Example") \
    .getOrCreate()

3. Read Dataset

In [7]:
df = spark.read.csv("Datasets.csv", header=True, inferSchema=True)
df.show()

+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|   Retailer|Retailer ID|Invoice Date|   Region|   State|    City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|
+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|Foot Locker|    1185732|    1/1/2020|Northeast|New York|New York|Men's Street Foot...|       $50.00 |     1,200|  $600,000 |       $300,000 |             50%|    In-store|
|Foot Locker|    1185732|    1/2/2020|Northeast|New York|New York|Men's Athletic Fo...|       $50.00 |     1,000|  $500,000 |       $150,000 |             30%|    In-store|
|Foot Locker|    1185732|    1/3/2020|Northeast|New York|New York|Women's Street Fo...|       $40.00 |     1,000|  $400,000 |       $14

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


4. Filter Operation (Conditions)

In [12]:
# Example: Retailer ID > 1185732
df.filter(df['Retailer ID'] > 1185732).show()

+-------------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|     Retailer|Retailer ID|Invoice Date|   Region|   State|    City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|
+-------------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|Sports Direct|    1197831|   7/19/2020|Northeast|New York|New York|Men's Street Foot...|       $25.00 |       900|  $225,000 |        $78,750 |             35%|      Outlet|
|Sports Direct|    1197831|   7/20/2020|Northeast|New York|New York|Men's Athletic Fo...|       $35.00 |       900|  $315,000 |       $110,250 |             35%|      Outlet|
|Sports Direct|    1197831|   7/21/2020|Northeast|New York|New York|Women's Street Fo...|       $35.00 |       700|  $245,000

5. Logical Operators
✔ AND (&)

In [14]:
df.filter((df['Retailer ID'] > 1185732) & (df.Region == 'Northeast')).show()

+-------------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|     Retailer|Retailer ID|Invoice Date|   Region|   State|    City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|
+-------------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|Sports Direct|    1197831|   7/19/2020|Northeast|New York|New York|Men's Street Foot...|       $25.00 |       900|  $225,000 |        $78,750 |             35%|      Outlet|
|Sports Direct|    1197831|   7/20/2020|Northeast|New York|New York|Men's Athletic Fo...|       $35.00 |       900|  $315,000 |       $110,250 |             35%|      Outlet|
|Sports Direct|    1197831|   7/21/2020|Northeast|New York|New York|Women's Street Fo...|       $35.00 |       700|  $245,000

In [16]:
df.filter((df['Retailer ID'] > 1197831) | (df.Region == 'West')).show()

+---------+-----------+------------+------+----------+-------------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
| Retailer|Retailer ID|Invoice Date|Region|     State|         City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|
+---------+-----------+------------+------+----------+-------------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|West Gear|    1128299|   11/5/2020|  West|California|San Francisco|       Men's Apparel|       $55.00 |       575|  $316,250 |       $158,125 |             50%|      Outlet|
|West Gear|    1128299|   11/6/2020|  West|California|San Francisco|     Women's Apparel|       $50.00 |       775|  $387,500 |        $58,125 |             15%|      Outlet|
|West Gear|    1128299|   11/7/2020|  West|California|San Francisco|Men's Street Foot...|       $40.00 |       825|  $330,000

6. Comparison Operators

In [17]:
# Equal
df.filter(df.City == "Delhi").show()

# Not Equal
df.filter(df.City != "Delhi").show()

+--------+-----------+------------+------+-----+----+-------+--------------+----------+-----------+----------------+----------------+------------+
|Retailer|Retailer ID|Invoice Date|Region|State|City|Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|
+--------+-----------+------------+------+-----+----+-------+--------------+----------+-----------+----------------+----------------+------------+
+--------+-----------+------------+------+-----+----+-------+--------------+----------+-----------+----------------+----------------+------------+

+-----------+-----------+------------+---------+--------+--------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|   Retailer|Retailer ID|Invoice Date|   Region|   State|    City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|
+-----------+-----------+------------+---------+--------+--------

7. GroupBy

In [18]:
df.groupBy("City").count().show()

+-------------+-----+
|         City|count|
+-------------+-----+
|   Charleston|  288|
|      Phoenix|  216|
|        Omaha|  144|
|    Anchorage|  144|
|       Dallas|  216|
|   Manchester|  216|
| Philadelphia|  216|
|   Louisville|  144|
|  Los Angeles|  216|
| Indianapolis|  144|
|San Francisco|  216|
|Oklahoma City|  216|
|      Detroit|  144|
|     Portland|  360|
|       Albany|  144|
|        Boise|  216|
|     Cheyenne|  144|
|    St. Louis|  144|
|   Birmingham|  216|
|   Burlington|  216|
+-------------+-----+
only showing top 20 rows


8. Aggregated Functions

In [22]:
from pyspark.sql.functions import regexp_replace
from pyspark.sql.types import DoubleType

# Clean the 'Total Sales' column by removing '$' and ',' and casting to DoubleType
df = df.withColumn("Total Sales",
                   regexp_replace(df["Total Sales"], "\\$|,", "").cast(DoubleType()))

print("Cleaned 'Total Sales' column and converted to DoubleType.")
df.printSchema()

Cleaned 'Total Sales' column and converted to DoubleType.
root
 |-- Retailer: string (nullable = true)
 |-- Retailer ID: integer (nullable = true)
 |-- Invoice Date: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Price per Unit: string (nullable = true)
 |-- Units Sold: string (nullable = true)
 |-- Total Sales: double (nullable = true)
 |-- Operating Profit: string (nullable = true)
 |-- Operating Margin: string (nullable = true)
 |-- Sales Method: string (nullable = true)



**Reasoning**:
Now that the 'Total Sales' column has been cleaned and converted to a numeric type, I will perform the requested aggregations (average, sum, maximum, and minimum) grouped by 'City' on this column.



In [23]:
from pyspark.sql.functions import avg, sum, max, min

df.groupBy("City").agg(
    avg("Total Sales").alias("Average_Sales"),
    sum("Total Sales").alias("Total_Sales_Sum"),
    max("Total Sales").alias("Max_Sales"),
    min("Total Sales").alias("Min_Sales")
).show()

+-------------+------------------+---------------+---------+---------+
|         City|     Average_Sales|Total_Sales_Sum|Max_Sales|Min_Sales|
+-------------+------------------+---------------+---------+---------+
|   Charleston|138801.37847222222|    3.9974797E7| 752500.0|    560.0|
|      Phoenix| 73065.83796296296|    1.5782221E7| 367500.0|   1599.0|
|        Omaha|         41173.875|      5929038.0| 247500.0|      0.0|
|    Anchorage|102452.10416666667|    1.4753103E7| 450000.0|   1764.0|
|       Dallas| 96772.51851851853|    2.0902864E7| 542500.0|    644.0|
|   Manchester| 75979.93981481482|    1.6411667E7| 468750.0|   1485.0|
| Philadelphia|47951.476851851854|    1.0357519E7| 341250.0|    256.0|
|   Louisville| 69950.33333333333|    1.0072848E7| 341250.0|    972.0|
|  Los Angeles|118680.15277777778|    2.5634913E7| 520000.0|   1107.0|
| Indianapolis| 61362.48611111111|      8836198.0| 360000.0|    589.0|
|San Francisco| 159903.7962962963|     3.453922E7| 700000.0|   5824.0|
|Oklah

9. format_number & Alias

In [32]:
from pyspark.sql.functions import format_number, avg

df.groupBy("City").agg(
    format_number(avg("Total Sales"), 2).alias("Avg_Total_Sales")
).show()

+-------------+---------------+
|         City|Avg_Total_Sales|
+-------------+---------------+
|   Charleston|     138,801.38|
|      Phoenix|      73,065.84|
|        Omaha|      41,173.88|
|    Anchorage|     102,452.10|
|       Dallas|      96,772.52|
|   Manchester|      75,979.94|
| Philadelphia|      47,951.48|
|   Louisville|      69,950.33|
|  Los Angeles|     118,680.15|
| Indianapolis|      61,362.49|
|San Francisco|     159,903.80|
|Oklahoma City|      49,170.06|
|      Detroit|     129,343.28|
|     Portland|      84,849.03|
|       Albany|     169,637.53|
|        Boise|      89,244.81|
|     Cheyenne|     129,010.53|
|    St. Louis|      67,244.90|
|   Birmingham|      81,636.22|
|   Burlington|      66,448.72|
+-------------+---------------+
only showing top 20 rows


10. Sort Data using orderBy

In [35]:
# Ascending
df.orderBy("Total Sales").show()

# Descending
df.orderBy(df["Total Sales"].desc()).show()

+-----------+-----------+------------+---------+------------+------------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|   Retailer|Retailer ID|Invoice Date|   Region|       State|        City|             Product|Price per Unit|Units Sold|Total Sales|Operating Profit|Operating Margin|Sales Method|
+-----------+-----------+------------+---------+------------+------------+--------------------+--------------+----------+-----------+----------------+----------------+------------+
|Foot Locker|    1185732|    6/5/2021|  Midwest|    Nebraska|       Omaha|Women's Athletic ...|       $35.00 |         0|        0.0|             $0 |             40%|      Outlet|
|Foot Locker|    1185732|   6/11/2021|  Midwest|    Nebraska|       Omaha|Women's Athletic ...|       $30.00 |         0|        0.0|             $0 |             40%|      Outlet|
|Foot Locker|    1185732|    6/5/2021|  Midwest|    Nebraska|       Omaha|Women's Athletic ...|